# Dioptra — Component Retraining Ablation Suite

This notebook executes the **true architectural retraining ablations** on Kaggle dual NVIDIA T4 GPUs.
Each model is trained **from Epoch 0** across all 24 epochs on the full 18,464 TartanAir training frames to isolate the individual contributions of Dioptra's core geometric mechanisms:

| Ablation Variant | CLI Flag | Ray Positional Encoding | Attention Mechanism | Camera Training | Purpose |
|---|---|---|---|---|---|
| **1. No ARA (Vanilla Attention)** | `--ablation trivision-no-irer` | Full Trivision Triplet | Vanilla Self-Attention (ARA Gate = 0.0) | Fixed $\mathbf{K}$ | Isolates the Angular Residual Attention inductive bias |
| **2. Center-Ray PE** | `--ablation center-ray` | Single Center Ray | Angular Residual Attention | Fixed $\mathbf{K}$ | Isolates the 3-ray triplet aperture & corner geometry |
| **3. Canonical 2D ViT** | `--ablation 2d-vit` | 2D Coordinate Grid (No Rays) | Vanilla Self-Attention | Camera-Blind | Evaluates standard 2D vision transformer baseline |

**Prerequisites (Right Sidebar)**:
1. **Settings → Accelerator**: GPU T4 ×2 (or P100)
2. **+ Add Input**:
   - Code dataset (contains `dioptra.py` or `dioptra.py`)
   - **DASVO TartanAir RGB-D Validation Split** (`pandrii000/dasvo-tartanair-rgb-d-validation-split`)


In [ ]:
# [1] Verify GPU allocation and environment
!nvidia-smi
import torch
print("CUDA available :", torch.cuda.is_available())
print("Device count   :", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"Device {i}       : {torch.cuda.get_device_name(i)}")

In [ ]:
# [2] Locate script, verify dataset mount, and prepare execution environment
import glob, os, shutil

scripts = sorted(glob.glob('/kaggle/input/**/dioptra.py', recursive=True)) + \
          sorted(glob.glob('/kaggle/input/**/dioptra.py', recursive=True))
assert scripts, 'dioptra.py not found — please attach the code dataset!'
input_script = scripts[-1]
print(f"Found mounted script: {input_script}")

# Ensure writable working copy
working_script = "/kaggle/working/dioptra.py"
shutil.copyfile(input_script, working_script)
SCRIPT = working_script
os.environ["SCRIPT_PATH"] = SCRIPT

# Verify TartanAir mount
def list_mounts(base='/kaggle/input'):
    out = []
    if not os.path.isdir(base): return out
    for name in sorted(os.listdir(base)):
        p = os.path.join(base, name)
        if not os.path.isdir(p): continue
        if name == 'datasets':
            for owner in sorted(os.listdir(p)):
                op = os.path.join(p, owner)
                if os.path.isdir(op):
                    for slug in sorted(os.listdir(op)):
                        if os.path.isdir(os.path.join(op, slug)):
                            out.append(f'datasets/{owner}/{slug}')
        else:
            out.append(name)
    return out

mounts = list_mounts()
print('Mounted inputs:', mounts)
assert any(('tartanair' in m.lower()) or ('dasvo' in m.lower()) for m in mounts), (
    'The DASVO TartanAir dataset is NOT attached to this notebook!\n'
    'Right sidebar → Input → "+ Add Input" → search "dasvo-tartanair-rgb-d-validation-split"'
)
print('Dataset mounted and ready ✓')

In [ ]:
# [3] Run architectural unit tests across all ablation configurations
!python "$SCRIPT_PATH" --test

### Choose which ablation to train:
- **Recommended**: Run **Cell [4A]** for the primary architectural ablation (**Without ARA**, ~6 hours).
- You can run **Cell [4B]** for Center-Ray PE or **Cell [4C]** for 2D-ViT in a separate notebook in parallel.
- *Note*: Cells [5] and [6] work automatically even if only ONE ablation has been trained!

In [ ]:
# [4A] ABLATION 1: Retrain Without Angular Residual Attention (Trivision PE + Vanilla Attention)
# Trains the network across 24 epochs with zero geometric attention bias (standard transformer attention).
!python -u "$SCRIPT_PATH" \
    --train auto \
    --ablation trivision-no-irer \
    --split-mode cross_env \
    --batch-size 4 \
    --epochs 24 \
    --output-dir /kaggle/working/outputs_no_irer

In [ ]:
# [4B] ABLATION 2: Retrain With Single Center-Ray PE (No Aperture Triplets)
# Replaces the 3-ray bundle with a replicated principal center ray.
!python -u "$SCRIPT_PATH" \
    --train auto \
    --ablation center-ray \
    --split-mode cross_env \
    --batch-size 4 \
    --epochs 24 \
    --output-dir /kaggle/working/outputs_center_ray

In [ ]:
# [4C] ABLATION 3: Retrain Canonical 2D Vision Transformer (No 3D Ray PE, No ARA)
# Standard 2D ViT baseline (learned coordinate embedding + vanilla attention).
!python -u "$SCRIPT_PATH" \
    --train auto \
    --ablation 2d-vit \
    --split-mode cross_env \
    --batch-size 4 \
    --epochs 24 \
    --output-dir /kaggle/working/outputs_2d_vit

In [ ]:
# [5] Comprehensive Evaluation & Automated Summary Table Generation
# Automatically detects whichever ablation checkpoints exist and evaluates them.
# Works whether you ran 1, 2, or all 3 ablations!
import os, glob

runs = [
    ("No ARA (Trained from Scratch)", "/kaggle/working/outputs_no_irer/checkpoint.pt", "trivision-no-irer"),
    ("Center-Ray (Trained from Scratch)", "/kaggle/working/outputs_center_ray/checkpoint.pt", "center-ray"),
    ("Canonical 2D ViT (Trained from Scratch)", "/kaggle/working/outputs_2d_vit/checkpoint.pt", "2d-vit"),
]

found_any = False
for name, ckpt_p, abl_flag in runs:
    if not os.path.exists(ckpt_p):
        print(f"[-] Checkpoint for {name} not found (skipped).")
        continue
    found_any = True
    print(f"\n{'='*70}")
    print(f"  Evaluating: {name}")
    print(f"  Checkpoint: {ckpt_p}")
    print(f"{'='*70}")
    !python "$SCRIPT_PATH" --train auto --ablation "$abl_flag" --evaluate "$ckpt_p"

if not found_any:
    print("\nNo ablation checkpoints found in /kaggle/working yet.")
    print("Run one of Cells [4A], [4B], or [4C] above first.")
else:
    print("\nEvaluation complete for all available ablation models! ✓")

In [ ]:
# [6] Dynamically Package Available Ablation Checkpoints for Download
# Only zips directories that actually exist — 100% error-free!
import os

candidate_dirs = [
    "/kaggle/working/outputs_no_irer",
    "/kaggle/working/outputs_center_ray",
    "/kaggle/working/outputs_2d_vit",
]
existing_dirs = [d for d in candidate_dirs if os.path.isdir(d)]

if existing_dirs:
    zip_target = "/kaggle/working/tesseract_ablations.zip"
    dirs_str = " ".join(existing_dirs)
    print(f"Zipping {len(existing_dirs)} available ablation directory(ies):")
    for d in existing_dirs: print(f"  + {d}")
    os.system(f"zip -r {zip_target} {dirs_str}")
    sz_mb = os.path.getsize(zip_target) / (1024 * 1024)
    print(f"\n✓ Successfully created {zip_target} ({sz_mb:.2f} MB)")
    print("You can now download it from the right sidebar Output tab.")
else:
    print("No ablation output directories found to zip yet.")